# **01_Model_Training: Baseline Models for Phishing Detection**

This notebook trains multiple **baseline classification models** on the preprocessed phishing dataset.

**Main steps:**
1. **Data Preparation**
   - Load the preprocessed dataset
   - Split the data into training and test sets

2. **Baseline Model Training**
   - Train a **Random Forest** classifier
   - Train a **Gradient Boosting** classifier
   - Train a **Logistic Regression** classifier

3. **Model Persistence**
   - Save trained models and search objects for later comparison and evaluation

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import randint, uniform
from joblib import dump
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, RandomizedSearchCV, GridSearchCV

---

# **Data Preparation**

## Load Dataset

Load the preprocessed phishing dataset into a DataFrame and inspect its structure.

In [3]:
DATASET_PATH = "../data/phishing_dataset.csv"
df = pd.read_csv(DATASET_PATH)

df.head()

,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,qty_exclamation_url,qty_space_url,...,qty_ip_resolved,qty_nameservers,qty_mx_servers,ttl_hostname,tls_ssl_certificate,qty_redirects,url_google_index,domain_google_index,url_shortened,phishing
0,3,0,0,1,0,0,0,0,0,0,...,1,2,0,892,0,0,0,0,0,1
1,5,0,1,3,0,3,0,2,0,0,...,1,2,1,9540,1,0,0,0,0,1
2,2,0,0,1,0,0,0,0,0,0,...,1,2,3,589,1,0,0,0,0,0
3,4,0,2,5,0,0,0,0,0,0,...,1,2,0,292,1,0,0,0,0,1
4,2,0,0,0,0,0,0,0,0,0,...,1,2,1,3597,0,1,0,0,0,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88647 entries, 0 to 88646
Columns: 112 entries, qty_dot_url to phishing
dtypes: float64(1), int64(111)
memory usage: 75.7 MB


In [16]:
list(df.columns)

['qty_dot_url',
 'qty_hyphen_url',
 'qty_underline_url',
 'qty_slash_url',
 'qty_questionmark_url',
 'qty_equal_url',
 'qty_at_url',
 'qty_and_url',
 'qty_exclamation_url',
 'qty_space_url',
 'qty_tilde_url',
 'qty_comma_url',
 'qty_plus_url',
 'qty_asterisk_url',
 'qty_hashtag_url',
 'qty_dollar_url',
 'qty_percent_url',
 'qty_tld_url',
 'length_url',
 'qty_dot_domain',
 'qty_hyphen_domain',
 'qty_underline_domain',
 'qty_slash_domain',
 'qty_questionmark_domain',
 'qty_equal_domain',
 'qty_at_domain',
 'qty_and_domain',
 'qty_exclamation_domain',
 'qty_space_domain',
 'qty_tilde_domain',
 'qty_comma_domain',
 'qty_plus_domain',
 'qty_asterisk_domain',
 'qty_hashtag_domain',
 'qty_dollar_domain',
 'qty_percent_domain',
 'qty_vowels_domain',
 'domain_length',
 'domain_in_ip',
 'server_client_domain',
 'qty_dot_directory',
 'qty_hyphen_directory',
 'qty_underline_directory',
 'qty_slash_directory',
 'qty_questionmark_directory',
 'qty_equal_directory',
 'qty_at_directory',
 'qty_and_dir

## Prepare Features and Labels, and Split Dataset

Separate the dataset into **features (`X`)** and **target (`y`)**, and display the total number of phishing cases to understand class distribution.  

Next, split the data into **training and test sets** with an 80-20 split, using **stratification** to preserve the class balance.  

Finally, save the train and test set to disk so it can be reused later in the evaluation notebook without rerunning the split.

In [6]:
LABEL_COLUMN = "phishing"
X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN]

print(f"Total Phising Cases: {y.values.sum()}")

Total Phising Cases: 30647


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

---

# **Baseline Model Training**

## Random Forest

Create a **pipeline** containing a `RandomForestClassifier` to allow easy integration with **scikit-learn tools** like `RandomizedSearchCV`.  

Define the **hyperparameter search space** for randomized search, including:
- `n_estimators` : Number of trees in the forest.
- `max_depth` : Maximum depth of each tree.
- `min_samples_split` : Minimum number of samples required to split a node.
- `min_samples_leaf` : Minimum number of samples required at a leaf node.
- `max_features` : Maximum number of features considered at each split.
- `class_weight` : Class weighting to handle potential class imbalance.

This search space allows the `RandomizedSearchCV` to explore multiple combinations efficiently and find a robust model.

In [ ]:
rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])

In [ ]:
rf_pipeline.get_params()

{'memory': None,
 'steps': [('model', RandomForestClassifier(random_state=42))],
 'transform_input': None,
 'verbose': False,
 'model': RandomForestClassifier(random_state=42),
 'model__bootstrap': True,
 'model__ccp_alpha': 0.0,
 'model__class_weight': None,
 'model__criterion': 'gini',
 'model__max_depth': None,
 'model__max_features': 'sqrt',
 'model__max_leaf_nodes': None,
 'model__max_samples': None,
 'model__min_impurity_decrease': 0.0,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__min_weight_fraction_leaf': 0.0,
 'model__monotonic_cst': None,
 'model__n_estimators': 100,
 'model__n_jobs': None,
 'model__oob_score': False,
 'model__random_state': 42,
 'model__verbose': 0,
 'model__warm_start': False}

In [ ]:
rf_param_dist = {
    'model__n_estimators': randint(100, 501),
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': randint(2, 11),
    'model__min_samples_leaf': randint(1, 6),
    'model__max_features': ['sqrt', 'log2', None],
    "model__class_weight": [None, 'balanced'] + [{0:1, 1:w} for w in np.linspace(1.2, 2.0, 5)]
}

Use **Stratified K-Fold cross-validation** to evaluate the model during hyperparameter search.  

- **Stratification** ensures that each fold preserves the **class distribution** of the target variable (phishing vs non-phishing).  
- `n_splits=5` divides the training data into 5 folds.  
- `shuffle=True` ensures the data is randomly shuffled before splitting to reduce bias.  
- `random_state=42` guarantees reproducibility of the splits.

In [23]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Use `RandomizedSearchCV` to find the best combination of hyperparameters for the Random Forest pipeline.  

Key settings:
- `estimator` : The pipeline containing the `RandomForestClassifier`.
- `param_distributions` : The hyperparameter search space defined earlier.
- `n_iter=30` : Randomly sample 30 hyperparameter combinations.
- `scoring='f1'` : Optimize for F1-score, which balances precision and recall for phishing detection.
- `cv=skf` : Use the previously defined Stratified K-Fold cross-validation.
- `n_jobs=-1` : Use all available CPU cores for parallel computation.
- `random_state=42` : Ensure reproducibility.
- `refit=True` : After search, automatically refit the best model on the full training set.

In [ ]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    random_state=42,
    refit=True
)

Fit the `RandomizedSearchCV` pipeline on the training data to find the best hyperparameters and train the Random Forest classifier.

After training, save the entire `RandomizedSearchCV` object using `joblib` to the `models/` folder.
This preserves the **best estimator, cross-validation results, and hyperparameter search metadata**, which will be used later in the evaluation notebook.

In [25]:
rf_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__class_weight': [None, 'balanced', ...], 'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': <scipy.stats....0020F6483C950>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the

In [ ]:
dump(rf_search, "../models/search/random_forest_random_search.joblib")

['../models/rf_random_search.joblib']

## Gradient Boosting

Create a **pipeline** containing a `GradientBoostingClassifier` to allow seamless integration with **scikit-learn tools** like `RandomizedSearchCV`.  

Define the **hyperparameter search space** for randomized search, including:
- `n_estimators` : Number of boosting stages to fit.
- `learning_rate` : Shrinks the contribution of each tree; balances learning speed and accuracy.
- `max_depth` : Maximum depth of each regression tree.
- `min_samples_split` : Minimum number of samples required to split an internal node.
- `min_samples_leaf` : Minimum number of samples required to be at a leaf node.
- `subsample` : Fraction of samples used for fitting each base learner; helps prevent overfitting.

This search space allows the `RandomizedSearchCV` to explore multiple parameter combinations efficiently and find a robust Gradient Boosting model.

In [ ]:
gb_pipeline = Pipeline([
    ('model', GradientBoostingClassifier(random_state=42))
])

In [ ]:
gb_pipeline.get_params()

{'memory': None,
 'steps': [('model', GradientBoostingClassifier(random_state=42))],
 'transform_input': None,
 'verbose': False,
 'model': GradientBoostingClassifier(random_state=42),
 'model__ccp_alpha': 0.0,
 'model__criterion': 'friedman_mse',
 'model__init': None,
 'model__learning_rate': 0.1,
 'model__loss': 'log_loss',
 'model__max_depth': 3,
 'model__max_features': None,
 'model__max_leaf_nodes': None,
 'model__min_impurity_decrease': 0.0,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__min_weight_fraction_leaf': 0.0,
 'model__n_estimators': 100,
 'model__n_iter_no_change': None,
 'model__random_state': 42,
 'model__subsample': 1.0,
 'model__tol': 0.0001,
 'model__validation_fraction': 0.1,
 'model__verbose': 0,
 'model__warm_start': False}

In [ ]:
gb_param_dist = {
    "model__n_estimators": randint(100, 501),
    "model__learning_rate": uniform(0.01, 0.19),
    "model__max_depth": randint(2, 5),
    "model__min_samples_split": randint(2, 11),
    "model__min_samples_leaf": randint(1, 6),
    "model__subsample": uniform(0.7, 0.3),
}

Use `RandomizedSearchCV` to find the best combination of hyperparameters for the Gradient Boosting pipeline.  

Key settings:
- `estimator` : The pipeline containing the `GradientBoostingClassifier`.
- `param_distributions` : The hyperparameter search space defined earlier.
- `n_iter=50` : Randomly sample 50 hyperparameter combinations.
- `scoring='f1'` : Optimize for F1-score, which balances precision and recall for phishing detection.
- `cv=skf` : Use the previously defined Stratified K-Fold cross-validation.
- `n_jobs=-1` : Use all available CPU cores for parallel computation.
- `random_state=42` : Ensure reproducibility.
- `refit=True` : After search, automatically refit the best model on the full training set.

In [ ]:
gb_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    random_state=42,
    refit=True
)

Fit the `RandomizedSearchCV` pipeline on the training data to find the best hyperparameters and train the Gradient Boosting classifier.

After training, save the entire `RandomizedSearchCV` object using `joblib` to the `models/` folder.
This preserves the **best estimator, cross-validation results, and hyperparameter search metadata**, which will be used later in the evaluation notebook.

In [8]:
gb_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': <scipy.stats....001C744B59010>, 'model__max_depth': <scipy.stats....001C744B2CB90>, 'model__min_samples_leaf': <scipy.stats....001C744A1FCE0>, 'model__min_samples_split': <scipy.stats....001C744B2D310>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across c

In [ ]:
dump(gb_search, "../models/search/gradient_boosting_random_search.joblib")

['../models/gb_random_search.joblib']

## Logistic Regression

Create a **pipeline** containing a `StandardScaler` and a `LogisticRegression` model to allow seamless integration with **scikit-learn tools** like `GridSearchCV`.  

`LogisticRegression` is configured with:
- `solver='saga'` : Efficient solver that supports both L1 and L2 regularization.
- `tol=1e-3` : Tolerance for stopping criteria; determines when the optimization converges.
- `max_iter=1000` : Maximum number of iterations for the solver to converge.
- `random_state=42` : Ensures reproducibility of results.

Define the **hyperparameter grid** for grid search, including:
- `model__C` : Regularization strength; smaller values imply stronger regularization.
- `model__l1_ratio` : Balances L1 (lasso) and L2 (ridge) regularization; 0=L2 only, 1=L1 only.
- `model__class_weight` : Class weighting to handle potential class imbalance; can be `None`, `balanced`, or custom weights `{0:1, 1:w}`.

This grid allows `GridSearchCV` to systematically explore combinations of hyperparameters and find the best Logistic Regression model.

In [31]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="saga",
        tol=1e-3,
        max_iter=1000,
        random_state=42
    ))
])

In [32]:
lr_pipeline.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('model',
   LogisticRegression(max_iter=1000, random_state=42, solver='saga', tol=0.001))],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'model': LogisticRegression(max_iter=1000, random_state=42, solver='saga', tol=0.001),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'model__C': 1.0,
 'model__class_weight': None,
 'model__dual': False,
 'model__fit_intercept': True,
 'model__intercept_scaling': 1,
 'model__l1_ratio': 0.0,
 'model__max_iter': 1000,
 'model__n_jobs': None,
 'model__penalty': 'deprecated',
 'model__random_state': 42,
 'model__solver': 'saga',
 'model__tol': 0.001,
 'model__verbose': 0,
 'model__warm_start': False}

In [33]:
lr_param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0, 1],
    'model__class_weight': [None, 'balanced'] + [{0:1, 1:w} for w in np.linspace(1.2, 2.0, 5)]
}

Use `GridSearchCV` to find the best combination of hyperparameters for the `LogisticRegression` pipeline.  

Key settings:
- `estimator` : The pipeline containing the `LogisticRegression` model.
- `param_grid` : The hyperparameter grid defined earlier.
- `scoring='f1'` : Optimize for F1-score, which balances precision and recall for phishing detection.
- `cv=skf` : Use the previously defined Stratified K-Fold cross-validation.
- `n_jobs=-1` : Use all available CPU cores for parallel computation.
- `refit=True` : After search, automatically refit the best model on the full training set.

In [ ]:
lr_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    refit=True
)

Fit the `GridSearchCV` pipeline on the training data to find the best hyperparameters and train the Logistic Regression classifier.

After training, save the entire `GridSearchCV` object using `joblib` to the `models/` folder.
This preserves the **best estimator, cross-validation results, and hyperparameter search metadata**, which will be used later in the evaluation notebook.

In [35]:
lr_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=0.001))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__class_weight': [None, 'balanced', ...], 'model__l1_ratio': [0, 1]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and

In [ ]:
dump(lr_search, "../models/search/logistic_regression_grid_search.joblib")

['../models/lr_grid_search.joblib']